# 03 — Treinamento do Modelo

**Objetivo:** Treinar um classificador de câncer pulmonar usando Transfer Learning
com ResNet50 pré-treinada no ImageNet.

## Estratégia

1. **Fase 1 (Feature Extraction):** Backbone congelado, treina apenas a cabeça customizada.
2. **Fase 2 (Fine-Tuning):** Descongela as últimas camadas do backbone para ajuste fino.

Esta abordagem em duas fases evita que os pesos pré-treinados sejam destruídos por
gradientes grandes no início do treino.

## 1. Configuração

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..') / 'src'))

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
import matplotlib.pyplot as plt

from config import (
    TAMANHO_LOTE, EPOCAS, TAXA_APRENDIZADO, PESO_DECAIMENTO,
    PACIENCIA, SEMENTE_ALEATORIA, ARQUITETURA_MODELO,
    CAMINHO_MODELOS, NOMES_CLASSES,
)
from dataset import criar_dataloaders
from modelo import criar_modelo
from treinamento import treinar_modelo
from avaliacao import plotar_historico_treino

# Reprodutibilidade total
torch.manual_seed(SEMENTE_ALEATORIA)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEMENTE_ALEATORIA)

DISPOSITIVO = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {DISPOSITIVO}')
print(f'PyTorch: {torch.__version__}')

## 2. Carregando os Dados

In [ ]:
# num_workers=0 para compatibilidade com macOS em notebooks
loader_treino, loader_valid, loader_teste = criar_dataloaders(
    tamanho_lote=TAMANHO_LOTE,
    num_workers=0,
)

print(f'Batches de treino    : {len(loader_treino)}')
print(f'Batches de validação : {len(loader_valid)}')
print(f'Batches de teste     : {len(loader_teste)}')

## 3. Construindo o Modelo

In [ ]:
modelo = criar_modelo(
    arquitetura=ARQUITETURA_MODELO,
    congelar_backbone=True,   # Fase 1: backbone congelado
    dispositivo=DISPOSITIVO,
)

stats = modelo.contar_parametros()
print(f'Arquitetura   : {ARQUITETURA_MODELO}')
print(f'Total params  : {stats["total"]:,}')
print(f'Treináveis    : {stats["treinavel"]:,} ({stats["treinavel"]/stats["total"]*100:.1f}%)')
print(f'Congelados    : {stats["congelado"]:,}')

## 4. Pesos de Classe para Loss Balanceada

In [ ]:
from collections import Counter
from dataset import DatasetPulmao, obter_transformacoes
from config import CAMINHO_TREINO

ds_treino = DatasetPulmao(CAMINHO_TREINO, transformacoes=obter_transformacoes('treino'))
contagem  = Counter(ds_treino.rotulos)
total     = sum(contagem.values())

# Peso inversamente proporcional à frequência da classe
pesos_classe = torch.tensor(
    [total / (len(contagem) * contagem[i]) for i in sorted(contagem.keys())],
    dtype=torch.float32,
).to(DISPOSITIVO)

print('Pesos por classe:')
for nome, peso in zip(NOMES_CLASSES, pesos_classe.tolist()):
    print(f'  {nome:<40} : {peso:.4f}')

## 5. Fase 1 — Feature Extraction (backbone congelado)

In [ ]:
# Apenas os parâmetros da cabeça customizada são otimizados
criterio    = nn.CrossEntropyLoss(weight=pesos_classe)
otimizador  = optim.Adam(
    filter(lambda p: p.requires_grad, modelo.parameters()),
    lr=TAXA_APRENDIZADO * 10,   # LR maior na fase 1 — só a cabeça treina
    weight_decay=PESO_DECAIMENTO,
)
scheduler = ReduceLROnPlateau(
    otimizador, mode='min', factor=0.5, patience=3, verbose=True
)

print('Iniciando Fase 1: Feature Extraction...')
historico_fase1 = treinar_modelo(
    modelo=modelo,
    loader_treino=loader_treino,
    loader_validacao=loader_valid,
    criterio=criterio,
    otimizador=otimizador,
    scheduler=scheduler,
    epocas=10,              # Poucas épocas na fase 1
    paciencia=5,
    dispositivo=DISPOSITIVO,
)

## 6. Fase 2 — Fine-Tuning (descongelando backbone)

In [ ]:
# Descongela as últimas 30 camadas do backbone para ajuste fino
modelo.descongelar_backbone(camadas=30)

stats = modelo.contar_parametros()
print(f'Parâmetros treináveis após descongelamento: {stats["treinavel"]:,}')

# LR menor para não destruir os pesos pré-treinados
otimizador_ft = optim.Adam(
    filter(lambda p: p.requires_grad, modelo.parameters()),
    lr=TAXA_APRENDIZADO,
    weight_decay=PESO_DECAIMENTO,
)
scheduler_ft = ReduceLROnPlateau(
    otimizador_ft, mode='min', factor=0.5, patience=3, verbose=True
)

print('\nIniciando Fase 2: Fine-Tuning...')
historico_fase2 = treinar_modelo(
    modelo=modelo,
    loader_treino=loader_treino,
    loader_validacao=loader_valid,
    criterio=criterio,
    otimizador=otimizador_ft,
    scheduler=scheduler_ft,
    epocas=EPOCAS,
    paciencia=PACIENCIA,
    dispositivo=DISPOSITIVO,
)

## 7. Visualização do Histórico

In [ ]:
# Concatena os históricos das duas fases
historico_completo = {
    chave: historico_fase1[chave] + historico_fase2[chave]
    for chave in historico_fase1
}

plotar_historico_treino(historico_completo, salvar=True)

# Melhor época
melhor_epoca = historico_completo['loss_validacao'].index(min(historico_completo['loss_validacao'])) + 1
print(f'\nMelhor época: {melhor_epoca}')
print(f'Melhor loss de validação  : {min(historico_completo["loss_validacao"]):.4f}')
print(f'Melhor acurácia de val    : {max(historico_completo["acuracia_validacao"]):.4f}')

## 8. Salvando o Histórico

In [ ]:
import json
from config import CAMINHO_REPORTS

with open(CAMINHO_REPORTS / 'historico_treinamento.json', 'w', encoding='utf-8') as f:
    json.dump(historico_completo, f, indent=2, ensure_ascii=False)

print('Histórico salvo em reports/historico_treinamento.json')
print('\nPróximo passo: abrir 04_avaliacao.ipynb para avaliar o modelo no conjunto de teste.')